In [ ]:
# 1) Création du venv
!python -m venv .venv

# 2) Installation des libs dans le venv via le pip du venv
!.venv\Scripts\python -m pip install --upgrade pip
!.venv\Scripts\pip install -r requirements.txt

# 🏢 Projet ML - Système de Prédiction RH

Ce notebook implémente un système complet de comparaison de modèles ML pour prédire l'attrition des employés (départ de l'entreprise).

## 📋 Contenu
1. **Prétraitement des données** - Extraction et fusion des fichiers CSV
2. **Chargement et préparation des données**
3. **Définition des modèles**
4. **Réglage des hyperparamètres (optionnel)**
5. **Entraînement et évaluation**
6. **Comparaison des modèles**
7. **Validation croisée**

## 🔧 ÉTAPE 0 - Prétraitement des Données Brutes

### Extraction du fichier ZIP

Les données sont dans un fichier `in_out_time.zip` qui contient plusieurs fichiers CSV.

In [ ]:
import shutil
import os

data_root="data/"

# Extraire le fichier ZIP s'il existe
if os.path.exists(data_root + 'in_out_time.zip'):
    print("📦 Extraction du fichier in_out_time.zip...")
    shutil.unpack_archive(data_root + 'in_out_time.zip', data_root)
    print("✅ Extraction terminée")
else:
    print("⚠️ Le fichier in_out_time.zip n'existe pas. Vérifiez que les fichiers CSV sont déjà extraits.")

### Fusion des fichiers CSV

Les données RH sont réparties dans 5 fichiers différents :
- `general_data.csv` - Données générales des employés
- `employee_survey_data.csv` - Enquête auprès des employés
- `manager_survey_data.csv` - Enquête auprès des managers
- `in_time.csv` - Heures d'arrivée
- `out_time.csv` - Heures de départ

Nous allons les fusionner via l'`EmployeeID`.

In [ ]:
import pandas as pd
import glob

# Vérifier la présence des fichiers
required_files = [data_root + 'general_data.csv', data_root + 'employee_survey_data.csv', data_root + 'manager_survey_data.csv', 
                  data_root + 'in_time.csv', data_root + 'out_time.csv']
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"❌ Fichiers manquants: {missing_files}")
    print("⚠️ Assurez-vous d'avoir extrait le fichier ZIP ou que les CSV sont présents.")
else:
    print("📊 Chargement des fichiers CSV...")
    
    # Charger les fichiers principaux
    df1 = pd.read_csv(data_root + 'general_data.csv')
    df2 = pd.read_csv(data_root + 'employee_survey_data.csv')
    df3 = pd.read_csv(data_root + 'manager_survey_data.csv')
    
    print(f"  - general_data.csv: {df1.shape}")
    print(f"  - employee_survey_data.csv: {df2.shape}")
    print(f"  - manager_survey_data.csv: {df3.shape}")
    
    # Charger les fichiers temporels
    df4 = pd.read_csv(data_root + 'in_time.csv')
    df5 = pd.read_csv(data_root + 'out_time.csv')
    
    print(f"  - in_time.csv: {df4.shape}")
    print(f"  - out_time.csv: {df5.shape}")
    
    # Renommer la première colonne des fichiers in/out time
    df4 = df4.rename(columns={'Unnamed: 0': 'EmployeeID'})
    df5 = df5.rename(columns={'Unnamed: 0': 'EmployeeID'})
    
    print("\n🔧 Création de features agrégées à partir des données temporelles...")
    print("   (Pour éviter l'explosion de mémoire avec 1M+ colonnes)")
    
    # Créer des features agrégées au lieu d'utiliser toutes les colonnes de dates
    def create_time_features(df_time, prefix='in'):
        """Créer des features agrégées à partir des heures d'arrivée/départ"""
        # Sélectionner seulement les colonnes de dates (pas EmployeeID)
        date_cols = [col for col in df_time.columns if col != 'EmployeeID']
        
        # Convertir en format datetime
        df_values = df_time[date_cols].apply(pd.to_datetime, errors='coerce')
        
        # Extraire l'heure en format numérique (heures + minutes/60)
        df_hours = df_values.apply(lambda x: x.dt.hour + x.dt.minute/60.0)
        
        # Calculer des statistiques agrégées
        features = pd.DataFrame()
        features['EmployeeID'] = df_time['EmployeeID']
        features[f'{prefix}_avg_hour'] = df_hours.mean(axis=1)  # Heure moyenne
        features[f'{prefix}_std_hour'] = df_hours.std(axis=1)   # Variabilité
        features[f'{prefix}_min_hour'] = df_hours.min(axis=1)   # Plus tôt
        features[f'{prefix}_max_hour'] = df_hours.max(axis=1)   # Plus tard
        features[f'{prefix}_missing_days'] = df_values.isna().sum(axis=1)  # Jours manquants
        
        return features
    
    # Créer les features pour in_time et out_time
    in_features = create_time_features(df4, prefix='arrival')
    out_features = create_time_features(df5, prefix='departure')
    
    # Fusionner les features temporelles
    time_features = in_features.merge(out_features, on='EmployeeID', how='inner')
    
    # Calculer le temps de travail moyen
    time_features['avg_work_hours'] = (
        time_features['departure_avg_hour'] - time_features['arrival_avg_hour']
    )
    
    print(f"  - Features temporelles créées: {time_features.shape[1]-1} colonnes")
    
    print("\n🔗 Fusion des données...")
    # Fusion en chaîne (2 par 2)
    resultat = df1.merge(df2, on='EmployeeID', how='inner') \
                  .merge(df3, on='EmployeeID', how='inner') \
                  .merge(time_features, on='EmployeeID', how='inner')
    
    # Suppression des colonnes inutiles
    colonnes_a_supprimer = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeID']
    colonnes_existantes = [col for col in colonnes_a_supprimer if col in resultat.columns]
    
    if colonnes_existantes:
        resultat = resultat.drop(colonnes_existantes, axis=1)
        print(f"  - Colonnes supprimées: {colonnes_existantes}")
    
    # Sauvegarder
    resultat.to_csv('Dataset_clean.csv', index=False)
    
    print(f"\n✅ Dataset fusionné et sauvegardé dans 'Dataset_clean.csv'")
    print(f"   Dimensions: {resultat.shape[0]} lignes × {resultat.shape[1]} colonnes")
    print(f"\n📋 Aperçu des colonnes:")
    print(f"   Colonnes générales: {list(df1.columns[:5])}...")
    print(f"   Features temporelles: {list(time_features.columns[1:6])}...")
    
    # Afficher les premières lignes
    print(f"\n📊 Aperçu des données:")
    display(resultat.head())
    
    # Vérifier les valeurs manquantes
    print(f"\n🔍 Valeurs manquantes par colonne:")
    missing = resultat.isnull().sum()
    if missing.sum() > 0:
        display(missing[missing > 0])
    else:
        print("   Aucune valeur manquante !")

## 2️⃣ Chargement et Préparation des Données

Nous allons utiliser le fichier `Dataset_clean.csv` créé lors du prétraitement.

In [ ]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modèles
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV

# Métriques
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix, 
    roc_curve, precision_recall_curve, average_precision_score
)

# Configuration des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Imports terminés")

## 1️⃣ Imports et Configuration

In [ ]:
def get_data_pipeline(csv_path, target_column='Attrition'):
    """
    Créer le pipeline de chargement et prétraitement des données avec split 70/15/15:
    1. Remplir les valeurs manquantes (médiane pour numérique, mode pour catégoriel)
    2. Transformer les données catégorielles avec OneHotEncoder
    3. Standardisation
    4. Split en 3 ensembles : Train (70%), Validation (15%), Test (15%)
    
    Args:
        csv_path: chemin vers le fichier CSV
        target_column: nom de la colonne cible (par défaut 'Attrition')
    """
    print(f"[Loader] Chargement des données depuis: {csv_path}")
    
    # 1. CHARGER LES DONNÉES
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
    except FileNotFoundError:
        raise FileNotFoundError(f"Fichier introuvable: {csv_path}")

    print(f"[Loader] Dataset chargé: {df.shape[0]} lignes × {df.shape[1]} colonnes")
    
    # Vérifier si la colonne cible existe
    if target_column not in df.columns:
        print(f"\n⚠️ La colonne '{target_column}' n'existe pas.")
        print(f"📋 Colonnes disponibles: {list(df.columns[:20])}")
        raise ValueError(f"Colonne cible '{target_column}' introuvable dans le dataset")
    
    # Séparer features X et target y
    blacklist = []  # Liste pour éviter les fuites de données si nécessaire
    cols_to_drop = [c for c in blacklist if c in df.columns]
    
    if cols_to_drop:
        df = df.drop(cols_to_drop, axis=1)
        print(f"[Loader] Colonnes supprimées: {cols_to_drop}")
    
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    
    # Convertir la cible en format binaire si nécessaire (Yes/No -> 1/0)
    if y.dtype == 'object':
        print(f"[Loader] Conversion de la variable cible en format binaire")
        y = y.map({'Yes': 1, 'No': 0})
        if y.isnull().any():
            print(f"⚠️ Valeurs non converties détectées dans la cible")

    # 2. SPLIT 70/15/15 (train/validation/test)
    print(f"\n[Loader] Split des données: 70% Train / 15% Validation / 15% Test")
    
    # Premier split : 70% train, 30% temp (validation + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    # Deuxième split : diviser les 30% en 15% validation et 15% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # 3. PIPELINE DE PRÉTRAITEMENT
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    print(f"[Loader] Features numériques: {len(numeric_features)}")
    print(f"[Loader] Features catégorielles: {len(categorical_features)}")

    return X_train, X_val, X_test, y_train, y_val, y_test, preprocessor

print("✅ Fonction de chargement définie (split 70/15/15)")

In [ ]:
# Charger les données avec split 70/15/15
print("🔄 Chargement des données RH...\n")

# Vérifier si le fichier existe
if not os.path.exists("Dataset_clean.csv"):
    print("❌ Le fichier 'Dataset_clean.csv' n'existe pas.")
    print("⚠️ Veuillez exécuter les cellules de prétraitement ci-dessus d'abord.")
    raise FileNotFoundError("Dataset_clean.csv introuvable. Exécutez d'abord le prétraitement.")

# Charger avec la colonne cible appropriée (Attrition pour les données RH)
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor = get_data_pipeline(
    "Dataset_clean.csv", 
    target_column='Attrition'  # Changez si votre colonne cible a un autre nom
)

print(f"\n📊 Dimensions des données (70/15/15):")
print(f"   - Train:      X={X_train.shape}, y={y_train.shape}")
print(f"   - Validation: X={X_val.shape}, y={y_val.shape}")
print(f"   - Test:       X={X_test.shape}, y={y_test.shape}")

# Afficher la distribution de la target
print(f"\n🎯 Distribution de la variable cible:")
print(f"\n📊 Train set:")
display(pd.DataFrame({
    'Count': y_train.value_counts(),
    'Proportion': y_train.value_counts(normalize=True)
}))

print(f"\n📊 Validation set:")
display(pd.DataFrame({
    'Count': y_val.value_counts(),
    'Proportion': y_val.value_counts(normalize=True)
}))

print(f"\n📊 Test set:")
display(pd.DataFrame({
    'Count': y_test.value_counts(),
    'Proportion': y_test.value_counts(normalize=True)
}))

## 3️⃣ Définition des Modèles

In [ ]:
def create_logistic_model(preprocessor):
    """Régression Logistique"""
    model = LogisticRegression(random_state=42, max_iter=1000)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_perceptron_model(preprocessor):
    """Perceptron avec calibration pour predict_proba"""
    base_model = Perceptron(random_state=42, max_iter=1000)
    calibrated_model = CalibratedClassifierCV(base_model, cv=3)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', calibrated_model)
    ])
    return pipeline

def create_decision_tree_model(preprocessor):
    """Arbre de Décision"""
    model = DecisionTreeClassifier(random_state=42)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_random_forest_model(preprocessor):
    """Forêt Aléatoire"""
    model = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=4)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    return pipeline

def create_svm_model(preprocessor):
    """SVM Linéaire avec calibration"""
    base_model = LinearSVC(random_state=42, max_iter=1000)
    calibrated_model = CalibratedClassifierCV(base_model, cv=3)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', calibrated_model)
    ])
    return pipeline

def create_naive_bayes_model(preprocessor):
    """Naive Bayes (Gaussian)"""
    # Créer un preprocessor modifié pour Naive Bayes (sparse_output=False)
    numeric_features = preprocessor.transformers[0][2]
    categorical_features = preprocessor.transformers[1][2]
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor_nb = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    model = GaussianNB()
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_nb),
        ('classifier', model)
    ])
    return pipeline

print("✅ Fonctions de création de modèles définies")

In [ ]:
# Créer tous les modèles
print("🔨 Création des modèles...\n")

competitors = {
    "Logistic Regression": create_logistic_model(preprocessor),
    "Perceptron": create_perceptron_model(preprocessor),
    "Decision Tree": create_decision_tree_model(preprocessor),
    "Random Forest": create_random_forest_model(preprocessor),
    "Support Vector Machine": create_svm_model(preprocessor),
    "Naive Bayes": create_naive_bayes_model(preprocessor)
}

print(f"✅ {len(competitors)} modèles créés:")
for name in competitors.keys():
    print(f"   - {name}")

## 4️⃣ Réglage des Hyperparamètres (Optionnel)

⚠️ Cette étape peut prendre du temps. Mettez `enable_tuning = True` pour activer.

In [ ]:
def get_param_grids():
    """Grilles de paramètres pour le tuning"""
    param_grids = {
        'Logistic Regression': {
            'classifier__C': [0.1, 1, 10],
            'classifier__max_iter': [1000, 5000]
        },
        'Decision Tree': {
            'classifier__max_depth': [5, 10, 15],
            'classifier__min_samples_split': [2, 5],
            'classifier__min_samples_leaf': [1, 2]
        },
        'Random Forest': {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [10, 15, 20],
            'classifier__min_samples_split': [2, 5]
        },
        'Naive Bayes': {
            'classifier__var_smoothing': [1e-9, 1e-8, 1e-7]
        },
        'Support Vector Machine': {
            'classifier__estimator__C': [0.1, 1, 10],
            'classifier__estimator__max_iter': [1000, 5000]
        },
        'Perceptron': {
            'classifier__estimator__alpha': [0.001, 0.01, 0.1],
            'classifier__estimator__max_iter': [1000, 5000]
        }
    }
    return param_grids

def tune_model_with_validation(model_pipeline, X_train, y_train, X_val, y_val, model_name, scoring='f1'):
    """
    Optimiser un modèle avec GridSearchCV en utilisant le set de validation.
    Plus rigoureux que la cross-validation car le test set reste complètement séparé.
    """
    param_grids = get_param_grids()
    
    if model_name not in param_grids:
        print(f"[Warning] Pas de grille définie pour {model_name}")
        return model_pipeline, {}, None
    
    param_grid = param_grids[model_name]
    
    print(f"\n[Tuning] Optimisation: {model_name}")
    print(f"  Combinaisons à tester: {np.prod([len(v) for v in param_grid.values()])}")
    
    # GridSearch avec validation manuelle sur le set de validation
    from sklearn.model_selection import PredefinedSplit
    
    # Créer un split qui utilise train pour entraîner et val pour valider
    # -1 = train, 0 = validation
    split_index = [-1] * len(X_train) + [0] * len(X_val)
    
    # Concaténer train et val temporairement pour GridSearchCV
    X_combined = pd.concat([X_train, X_val])
    y_combined = pd.concat([y_train, y_val])
    
    ps = PredefinedSplit(test_fold=split_index)
    
    grid_search = GridSearchCV(
        model_pipeline,
        param_grid,
        cv=ps,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_combined, y_combined)
    
    print(f"  ✅ Meilleurs params: {grid_search.best_params_}")
    print(f"  ✅ Score validation ({scoring}): {grid_search.best_score_:.4f}")
    
    # Réentraîner sur train uniquement avec les meilleurs paramètres
    best_model = grid_search.best_estimator_
    best_model.fit(X_train, y_train)
    
    return best_model, grid_search.best_params_, grid_search.best_score_

def tune_all_models(competitors, X_train, y_train, X_val, y_val, scoring='f1'):
    """Optimiser tous les modèles avec le set de validation"""
    tuned_competitors = {}
    tuning_results = {}
    
    print("\n" + "="*60)
    print("🔧 Début du réglage des hyperparamètres (avec set de validation)")
    print("="*60)
    
    for model_name, pipeline in competitors.items():
        best_pipeline, best_params, best_score = tune_model_with_validation(
            pipeline, X_train, y_train, X_val, y_val, model_name, scoring=scoring
        )
        tuned_competitors[model_name] = best_pipeline
        tuning_results[model_name] = (best_params, best_score)
    
    print("\n" + "="*60)
    print("✅ Réglage terminé")
    print("="*60)
    
    # Afficher le résumé
    summary_data = []
    for model_name, (best_params, best_score) in tuning_results.items():
        summary_data.append({
            'Model': model_name,
            'Validation Score (F1)': f"{best_score:.4f}" if best_score else "N/A",
            'Num Params Tuned': len(best_params) if best_params else 0
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n📊 Résumé du tuning (sur set de validation):")
    display(summary_df)
    
    return tuned_competitors, tuning_results

print("✅ Fonctions de tuning définies (avec split validation)")

In [ ]:
# Configuration du tuning
enable_tuning = True  # Mettre à True pour activer le tuning

if enable_tuning:
    print("🔧 Tuning activé - Optimisation sur le set de validation")
    competitors, tuning_results = tune_all_models(competitors, X_train, y_train, X_val, y_val, scoring='f1')
else:
    print("⏭️ Tuning désactivé. Utilisation des hyperparamètres par défaut.")
    print("💡 Conseil: Activez le tuning pour de meilleures performances !")

## 5️⃣ Entraînement et Évaluation des Modèles

⚠️ **Important**: L'évaluation se fait sur le **set de test** (15%) qui n'a jamais été vu pendant l'entraînement ou le tuning !

In [ ]:
def plot_confusion_matrix(model_name, cm):
    """Dessiner la matrice de confusion"""
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'])
    plt.title(f'Matrice de Confusion - {model_name}')
    plt.xlabel('Prédiction')
    plt.ylabel('Réalité')
    plt.tight_layout()
    plt.show()

def evaluate_model_detailed(model, X_test, y_test, model_name):
    """Évaluation détaillée d'un modèle"""
    print(f"\n{'='*60}")
    print(f"Évaluation: {model_name}")
    print(f"{'='*60}")
    
    # Prédictions
    y_pred = model.predict(X_test)
    
    # Rapport de classification
    print("\n📊 Rapport de classification:")
    print(classification_report(y_test, y_pred))
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n🔢 Matrice de confusion:")
    print(f"TP (VP): {cm[1][1]} | FP (FP): {cm[0][1]}")
    print(f"FN (FN): {cm[1][0]} | TN (VN): {cm[0][0]}")
    
    plot_confusion_matrix(model_name, cm)
    
    # Métriques
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    
    # Calculer AUC
    auc = "N/A"
    try:
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_test)
            if proba.ndim == 2:
                auc = roc_auc_score(y_test, proba[:, 1])
            else:
                auc = roc_auc_score(y_test, proba)
        elif hasattr(model, "decision_function"):
            scores = model.decision_function(X_test)
            if getattr(scores, "ndim", 1) == 2:
                auc = roc_auc_score(y_test, scores[:, 1])
            else:
                auc = roc_auc_score(y_test, scores)
    except Exception:
        pass
    
    return {
        "Model": model_name,
        "Accuracy": acc,
        "F1-Score": f1,
        "AUC": auc
    }

print("✅ Fonction d'évaluation définie")

In [ ]:
# Entraîner et évaluer tous les modèles sur le TEST SET
print("\n" + "="*60)
print("🚀 ENTRAÎNEMENT ET ÉVALUATION")
print("="*60)
print("📌 Entraînement: Train set (70%)")
print("📌 Évaluation: Test set (15%) - Jamais vu !")
print("="*60)

trained_models = {}
results_summary = []

for name, pipeline in competitors.items():
    print(f"\n🔄 Entraînement: {name}...")
    
    # Entraîner sur le train set uniquement
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    
    # Évaluation détaillée sur le TEST SET (complètement séparé)
    metrics = evaluate_model_detailed(pipeline, X_test, y_test, name)
    results_summary.append(metrics)

print("\n" + "="*60)
print("✅ ENTRAÎNEMENT ET ÉVALUATION TERMINÉS")
print("="*60)

## 6️⃣ Comparaison des Modèles

In [ ]:
# Table de comparaison
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values(by="F1-Score", ascending=False)

print("\n" + "#"*60)
print("🏆 COMPARAISON DES MODÈLES")
print("#"*60 + "\n")

display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print(f"\n🥇 Meilleur modèle: {best_model_name}")

In [ ]:
# Visualisation des performances
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
axes[0].barh(results_df['Model'], results_df['Accuracy'], color='skyblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Comparaison - Accuracy')
axes[0].set_xlim([0, 1])

# F1-Score
axes[1].barh(results_df['Model'], results_df['F1-Score'], color='lightgreen')
axes[1].set_xlabel('F1-Score')
axes[1].set_title('Comparaison - F1-Score')
axes[1].set_xlim([0, 1])

# AUC
auc_values = [v if isinstance(v, (int, float)) else 0 for v in results_df['AUC']]
axes[2].barh(results_df['Model'], auc_values, color='salmon')
axes[2].set_xlabel('AUC')
axes[2].set_title('Comparaison - AUC')
axes[2].set_xlim([0, 1])

plt.tight_layout()
plt.show()

## 7️⃣ Courbes ROC

In [ ]:
# Courbes ROC pour tous les modèles
plt.figure(figsize=(10, 7))

for name, pipeline in trained_models.items():
    try:
        if hasattr(pipeline, 'predict_proba'):
            proba = pipeline.predict_proba(X_test)
            scores = proba[:, 1] if proba.ndim == 2 else proba
        elif hasattr(pipeline, 'decision_function'):
            scores = pipeline.decision_function(X_test)
            if getattr(scores, 'ndim', 1) == 2:
                scores = scores[:, 1]
        else:
            continue

        fpr, tpr, _ = roc_curve(y_test, scores)
        auc_score = roc_auc_score(y_test, scores)
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})', linewidth=2)
    except Exception:
        continue

plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire', linewidth=1)
plt.title('Courbes ROC - Tous les Modèles', fontsize=14)
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8️⃣ Courbes Précision-Rappel

In [ ]:
# Courbes Précision-Rappel pour tous les modèles
plt.figure(figsize=(10, 7))

for name, pipeline in trained_models.items():
    try:
        if hasattr(pipeline, 'predict_proba'):
            proba = pipeline.predict_proba(X_test)
            scores = proba[:, 1] if proba.ndim == 2 else proba
        elif hasattr(pipeline, 'decision_function'):
            scores = pipeline.decision_function(X_test)
            scores = scores[:, 1] if getattr(scores, 'ndim', 1) == 2 else scores
        else:
            continue
        
        precision, recall, _ = precision_recall_curve(y_test, scores)
        ap = average_precision_score(y_test, scores)
        plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})", linewidth=2)
    except Exception:
        continue

plt.title('Courbes Précision-Rappel - Tous les Modèles', fontsize=14)
plt.xlabel('Rappel', fontsize=12)
plt.ylabel('Précision', fontsize=12)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9️⃣ Validation Croisée (5-Fold) - Optionnel

Cette validation se fait sur le **train set uniquement** pour évaluer la stabilité des modèles.

In [ ]:
def cross_validate_models(competitors, X, y, n_splits=5):
    """Validation croisée stratifiée pour tous les modèles"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_summary = []
    
    print("\n" + "="*60)
    print(f"🔄 Validation Croisée {n_splits}-Fold")
    print("="*60 + "\n")
    
    for name, pipeline in competitors.items():
        print(f"Validation: {name}...")
        auc_scores, f1_scores, acc_scores, ap_scores = [], [], [], []
        
        for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
            y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
            
            pipeline.fit(X_tr, y_tr)
            y_pred = pipeline.predict(X_te)
            
            acc_scores.append(accuracy_score(y_te, y_pred))
            f1_scores.append(f1_score(y_te, y_pred))
            
            # Calculer AUC et AP
            scores = None
            if hasattr(pipeline, 'predict_proba'):
                proba = pipeline.predict_proba(X_te)
                scores = proba[:, 1] if proba.ndim == 2 else proba
            elif hasattr(pipeline, 'decision_function'):
                scores = pipeline.decision_function(X_te)
                scores = scores[:, 1] if getattr(scores, 'ndim', 1) == 2 else scores
            
            if scores is not None:
                auc_scores.append(roc_auc_score(y_te, scores))
                ap_scores.append(average_precision_score(y_te, scores))
        
        cv_summary.append({
            'Model': name,
            'Acc(mean)': np.mean(acc_scores) if acc_scores else None,
            'F1(mean)': np.mean(f1_scores) if f1_scores else None,
            'AUC(mean)': np.mean(auc_scores) if auc_scores else None,
            'AP(mean)': np.mean(ap_scores) if ap_scores else None,
        })
    
    df = pd.DataFrame(cv_summary)
    print("\n" + "="*60)
    print("📊 Résultats de la Validation Croisée (moyennes)")
    print("="*60 + "\n")
    display(df)
    
    return df

# Exécuter la validation croisée
cv_results = cross_validate_models(competitors, X_train, y_train, n_splits=5)

## 🎯 Conclusion

Ce notebook a permis de :
1. ✅ Charger et prétraiter les données
2. ✅ Comparer 6 modèles de classification
3. ✅ Évaluer les performances avec plusieurs métriques
4. ✅ Visualiser les résultats avec ROC et PR curves
5. ✅ Valider avec cross-validation

Le meilleur modèle peut maintenant être utilisé pour la production !